# Authenticity EDA & Shortcut Checks
This notebook validates the AI-ArtBench dataset integration, demonstrating class balance and the removal of shortcuts (like resolution and JPEG format differences).

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import collections
from PIL import Image

Matplotlib is building the font cache; this may take a moment.


In [2]:
# Load metadata
metadata = pd.read_csv('../data/processed/metadata.csv')
metadata.head()

,image_id,filepath,source_dataset,style,movement,artist,genre,era,technique,mood,is_ai_generated,generator,width,height,file_hash,phash,split,mood_dist,mood_n_annotations,mood_agreement
0,ai_artbench_edward-weston_cabbage-leaf-1931,data/processed/images/ai_artbench_edward-westo...,ai_artbench,Symbolism,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,384,384,f83259a492f0fd2ec0da44f11994b02c,9a9b51397964a647,train,NaN,NaN,NaN
1,ai_artbench_6-461444567-930634,data/processed/images/ai_artbench_6-461444567-...,ai_artbench,High_Renaissance,NaN,NaN,NaN,NaN,NaN,NaN,True,AI_SD,384,384,e49e433228cfe28c4a652e54de3c1dca,c03c66d1596f2df0,train,NaN,NaN,NaN
2,ai_artbench_5-940247006-650857,data/processed/images/ai_artbench_5-940247006-...,ai_artbench,Realism,NaN,NaN,NaN,NaN,NaN,NaN,True,AI_SD,384,384,d55917ae81d918256694de83a45efe39,9c4b250cc3749bed,train,NaN,NaN,NaN
3,ai_artbench_theodor-severin-kittelsen_sommerfu...,data/processed/images/ai_artbench_theodor-seve...,ai_artbench,Realism,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,384,384,d3d548e05849702dfad5f62216a0af2f,80b0bf4be43fc338,train,NaN,NaN,NaN
4,ai_artbench_Duy_Huynh_Autumn_Introspection,data/processed/images/ai_artbench_Duy_Huynh_Au...,ai_artbench,Symbolism,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,384,384,5688b634980756b9ec5bce63606b880b,ed92126d8d7346e4,train,NaN,NaN,NaN


## Class Balance

In [3]:
# Human vs AI
print(metadata['is_ai_generated'].value_counts(normalize=True))

is_ai_generated
False    0.523583
True     0.476417
Name: proportion, dtype: float64


In [4]:
# Generators
print(metadata['generator'].value_counts())

generator
AI_SD    500
AI_LD    500
Name: count, dtype: int64


In [5]:
# Styles across generators
print(pd.crosstab(metadata['generator'], metadata['style']))

style      Art_Nouveau  Baroque  Expressionism  High_Renaissance  \
generator                                                          
AI_LD               50       50             50                50   
AI_SD               50       50             50                50   

style      Impressionism  Post_Impressionism  Realism  Romanticism  Symbolism  \
generator                                                                       
AI_LD                 50                  50       50           50         50   
AI_SD                 50                  50       50           50         50   

style      Ukiyo_e  
generator           
AI_LD           50  
AI_SD           50  


## Shortcut Checks (After Normalization)
We expect all images in our processed cache to have exactly the same size (`384x384`) and format, so the neural network can't cheat by looking at resolution.

In [6]:
sizes = collections.defaultdict(int)
for filepath in metadata['filepath']:
    with Image.open(f"../{filepath}") as img:
        sizes[img.size] += 1
print("Sizes found in normalized cache:", dict(sizes))

Sizes found in normalized cache: {(384, 384): 2099}


(Note: Before normalization, AI_SD was exclusively 768x768 and Human/AI_LD were 256x256. We've successfully standardized them to 384x384).

## Missing Label Masks Check

In [7]:
# We check how many rows are missing specific labels.
print("Rows missing 'era':", metadata['era'].isna().sum())
print("Rows missing 'mood':", metadata['mood'].isna().sum())

Rows missing 'era': 2000
Rows missing 'mood': 2000


## Sample Grid

In [8]:
# (In a real interactive notebook, we would plot a grid of images here.)
# For headless execution, we just confirm paths exist.
sample = metadata.sample(4)
for _, row in sample.iterrows():
    assert os.path.exists(f"../{row['filepath']}")
print("Sample images verified to exist.")

Sample images verified to exist.
